# 05 · BW + PLI + EM + MA: PEAF, SEAF, PDAF, SDAF (paper Figs. 9, 10, 11)

* **PEAF:** parallel EMD adaptive filter (Fig. 4a, Eq. 2)
* **SEAF:** staged EMD adaptive filter (Fig. 4b, Eq. 3)
* **PDAF:** all noises treated as one; one adaptive filter on the whole signal with the summed references
* **SDAF:** Thakor & Zhu [13] plus an MA layer (Fig. 5, no EMD)

Paper: **PEAF has the highest SNRimp** on every record at SNRin = 10 dB (Fig. 10) and at every SNRin (Fig. 11).

In [ ]:
from pathlib import Path
import sys, json

project_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

RESULTS = project_root / "results"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
METHOD_COLORS = {"PEAF": "#1f3b73", "SEAF": "#9aa83a", "PDAF": "#d62728", "SDAF": "#3cb9a0"}


In [ ]:
four = pd.read_csv(RESULTS / "four_methods.csv")
print("LMS setting:", json.loads((RESULTS / "lms_setting.json").read_text()))
LABEL = {"PEAF": "proposed emd+adap parallel (PEAF)", "SEAF": "proposed emd+adap staged (SEAF)",
         "PDAF": "adap parallel (PDAF)", "SDAF": "adap stage [13] (SDAF)"}

## Fig. 9: example (record 100, SNRin = 10 dB, PEAF, first 5000 samples)

In [ ]:
ex = np.load(RESULTS / "fig9_example.npz"); fs = int(ex["fs"])
sigs = [("original signal", ex["clean"]), ("signal + noise", ex["noisy"]), ("EMD denoised signal (PEAF)", ex["PEAF"])]
fig, axes = plt.subplots(3, 2, figsize=(16, 8), gridspec_kw={"width_ratios": [1.6, 1]})
for (name, s), (ax_t, ax_f) in zip(sigs, axes):
    ax_t.plot(s[:5000], lw=0.6); ax_t.set_title(name, fontsize=10); ax_t.set_ylabel("mV")
    f = np.fft.rfftfreq(len(s), 1 / fs)
    ax_f.plot(f, np.abs(np.fft.rfft(s - s.mean())) / len(s), lw=0.5)
    ax_f.set_xlim(0, 180); ax_f.set_title(name + " (spectrum, whole record)", fontsize=10)
axes[-1, 0].set_xlabel("sample"); axes[-1, 1].set_xlabel("f (Hz)")
plt.tight_layout(); plt.show()

### All four outputs on the same 3 s window (t = 600 s)

In [ ]:
w = slice(600 * fs, 603 * fs); t = np.arange(w.start, w.stop) / fs
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
for ax, m in zip(axes, ["PEAF", "SEAF", "PDAF", "SDAF"]):
    ax.plot(t, ex["noisy"][w], color="0.75", lw=0.6, label="signal + noise")
    ax.plot(t, ex["clean"][w], color="k", lw=1.2, label="original")
    ax.plot(t, ex[m][w], color=METHOD_COLORS[m], lw=0.9, label=m)
    ax.set_ylabel("mV"); ax.legend(fontsize=8, loc="upper right"); ax.set_title(LABEL[m], fontsize=10)
axes[-1].set_xlabel("time (s)")
plt.tight_layout(); plt.show()

## Fig. 10: SNRimp per record, SNRin = 10 dB

In [ ]:
d10 = four[four.SNRin_dB == 10].pivot(index="record", columns="method", values="SNRimp_dB")
fig, ax = plt.subplots(figsize=(13, 4.8))
for m in ["PEAF", "SEAF", "PDAF", "SDAF"]:
    ax.plot([str(r) for r in d10.index], d10[m], marker="o", color=METHOD_COLORS[m], label=LABEL[m])
ax.set_xlabel("signal"); ax.set_ylabel("SNRimp (dB)"); ax.set_title("SNRimp using different signal (SNRin = 10 dB)")
ax.legend(fontsize=8); plt.show()
t10 = d10.copy(); t10["best"] = d10.idxmax(axis=1); t10.loc["mean"] = d10.mean()
print("records where PEAF is best:", int((d10.idxmax(axis=1) == "PEAF").sum()), "of", len(d10))
t10.round(2)

## Fig. 11: SNRimp vs SNRin (mean over the 13 records)

In [ ]:
g = four.groupby(["SNRin_dB", "method"]).SNRimp_dB
mean11, std11 = g.mean().unstack(), g.std().unstack()
fig, axes = plt.subplots(1, 2, figsize=(16, 4.8), gridspec_kw={"width_ratios": [1.3, 1]})
for m in ["PEAF", "SEAF", "PDAF", "SDAF"]:
    axes[0].plot([str(s) for s in mean11.index], mean11[m], marker="o", color=METHOD_COLORS[m], label=LABEL[m])
axes[0].set_xlabel("SNRinput (dB)"); axes[0].set_ylabel("SNRimp (dB)"); axes[0].legend(fontsize=8)
axes[0].set_title("SNRimp under different SNRinput (mean of 13 records)")
data = [four[(four.SNRin_dB == s) & (four.method == m)].SNRimp_dB.values
        for s in mean11.index for m in ["PEAF", "SEAF", "PDAF", "SDAF"]]
pos = [i * 5 + k for i in range(len(mean11)) for k in range(4)]
bp = axes[1].boxplot(data, positions=pos, widths=0.8, patch_artist=True, showfliers=False)
for patch, k in zip(bp["boxes"], [k for _ in mean11.index for k in range(4)]):
    patch.set_facecolor(list(METHOD_COLORS.values())[k])
axes[1].set_xticks([i * 5 + 1.5 for i in range(len(mean11))], mean11.index)
axes[1].set_xlabel("SNRinput (dB)"); axes[1].set_ylabel("SNRimp (dB)"); axes[1].set_title("spread over the 13 records")
plt.tight_layout(); plt.show()
out = mean11.round(2); out["best"] = mean11.idxmax(axis=1); out

In [ ]:
print("SNRin levels where PEAF has the highest mean SNRimp:",
      list(mean11.index[mean11.idxmax(axis=1) == "PEAF"]), "of", list(mean11.index))